Import required libraries

In [1]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

batch_size = 2
sequence_length = 6
d_model = 32
num_heads = 4

x = torch.randn(batch_size, sequence_length, d_model)

print("Input shape:", x.shape)

Input shape: torch.Size([2, 6, 32])


Multi-Headed Attention Implementation

In [2]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):

        batch_size, seq_len, _ = x.shape

        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        Q = Q.reshape(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)

        K = K.reshape(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)

        V = V.reshape(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        weights = torch.softmax(scores, dim=-1)

        context = weights @ V

        context = context.transpose(1, 2).contiguous()
        context = context.reshape(batch_size, seq_len, self.d_model)

        output = self.W_O(context)

        return output, weights

Test MHA Implementation, since 32 embeding size is splitted amongst 4 heads, then we have the attention  weights to be

[2, 4, 6, 6]

representing [Batch, attn heads, seq_lenghts (6x6 matrix)]

In [4]:
mha = MultiHeadAttention(d_model=32, num_heads=4)

output, weights = mha(x)

print("Input:", x.shape)
print("Output:", output.shape)
print("Attention weights:", weights.shape)

Input: torch.Size([2, 6, 32])
Output: torch.Size([2, 6, 32])
Attention weights: torch.Size([2, 4, 6, 6])


Implementing Positional Encoding

In [5]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_length=100):
        super().__init__()

        position = torch.arange(max_length).float().unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_length, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # [1, max_length, d_model]
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

Testing Positional Encoding

In [6]:
vocab_size = 100
d_model = 32

embedding = nn.Embedding(vocab_size, d_model)
positional_encoding = PositionalEncoding(d_model, max_length=50)

token_ids = torch.tensor([
    [10, 25, 7, 42, 19]
])

token_embeddings = embedding(token_ids)
positioned_embeddings = positional_encoding(token_embeddings)

print("Token embeddings:", token_embeddings.shape)
print("After positional encoding:", positioned_embeddings.shape)

Token embeddings: torch.Size([1, 5, 32])
After positional encoding: torch.Size([1, 5, 32])


This is how positional represnetations change with token position.

In [7]:
# Same token embedding, placed at two different positions
same_token = embedding(torch.tensor([[10, 10, 10]]))

with_positions = positional_encoding(same_token)

print("Position 0:", with_positions[0, 0, :5])
print("Position 1:", with_positions[0, 1, :5])
print("Position 2:", with_positions[0, 2, :5])

Position 0: tensor([ 0.0229,  1.3753, -0.7553,  0.7751,  2.0404], grad_fn=<SliceBackward0>)
Position 1: tensor([ 0.8643,  0.9156, -0.2221,  0.6211,  2.3514], grad_fn=<SliceBackward0>)
Position 2: tensor([ 0.9322, -0.0409,  0.1469,  0.2065,  2.6315], grad_fn=<SliceBackward0>)
